# DMD Slice Analysis

Example usage of `pyNeuroDAP.slice` for DMD patch-clamp slice electrophysiology.

**Steps covered:**
1. Scan a `Results-*` folder and load the `cells_DMD` table
2. Query a specific spot's response traces
3. Build a tidy long-format DataFrame of all spots
4. Run `analyze_dmd_search` — per-depth metrics and summary figures
5. Run `analyze_dmd_search_pair` — paired search comparison

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyNeuroDAP as ndap

## Configuration

Edit `RESULTS_DIR` to point to your `Results-*` folder.

In [ ]:
RESULTS_DIR = (
    "/Volumes/Neurobio/MICROSCOPE/Shun/Project valence/Patch/"
    "from_DC_PLOM/SL406/Results-20260109"
)
SAVE_DIR = os.path.join(RESULTS_DIR, "python_output")

## 1. Load the `cells_DMD` table

In [ ]:
idx = ndap.index_results_folder(RESULTS_DIR)
print(f"cells MAT : {idx['cells_mat_path']}")
print(f"cell dirs : {list(idx['cell_dirs'].keys())}")
print(f"spots files: {len(idx['spots_files'])}")

In [ ]:
cells_df = ndap.load_cells_table(idx["cells_mat_path"])
print(f"{len(cells_df)} cells: {cells_df['Cell'].tolist()}")
for _, row in cells_df.iterrows():
    print(f"  Cell {row['Cell']}: epochs = {row['Epochs']}")
cells_df[["Cell", "Epochs", "Vhold"]]

## 2. Query a single spot's response

Retrieve all sweeps for **cell 3, search 0, depth 1, spot 0** (zero-indexed).
Use the `type` argument to return only a subset: `'traces'`, `'features'`, or `'meta'`.

In [ ]:
resp = ndap.get_spot_response(
    cells_df,
    cell=3,
    search_idx=0,
    depth=1,
    hotspot=0,
)

print("Meta:", resp["meta"])
print("\nTrace shapes:")
for key, traces in resp["traces"].items():
    for name, arr in traces.items():
        if isinstance(arr, np.ndarray):
            print(f"  {key}/{name}: {arr.shape}")

print("\nFeatures:", {k: v for k, v in resp["features"].items() if not isinstance(v, dict)})

In [ ]:
# Quick plot of the spot's opto trace(s)
opto = resp["traces"]["search_0"]["opto"]   # (n_sweeps, n_samples)
output_fs = resp["meta"]["output_fs"]
t_ms = (np.arange(opto.shape[1]) - resp["meta"]["event_sample"] + 1) / (output_fs / 1000)

fig, ax = plt.subplots(figsize=(8, 3))
for sweep in opto:
    ax.plot(t_ms, sweep, lw=0.8, alpha=0.6)
ax.axvline(0, color="r", ls="--", lw=1, label="stim onset")
ax.set_xlabel("Time from stim (ms)")
ax.set_ylabel("Current (pA)")
ax.set_title("Cell 3 — search 0 — depth 1 — spot 0")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Tidy long-format DataFrame of all spots

In [ ]:
long_df = ndap.results_to_long_dataframe(cells_df)
print(f"Shape: {long_df.shape}")
long_df.head(8)

In [ ]:
# Hotspot distribution across all cells / searches / depths
print(long_df.groupby(["cell", "depth"])["is_hotspot"].sum().unstack("depth").to_string())

## 4. Analyze a single search — `analyze_dmd_search`

Produces one summary figure per depth with:
- Tiled per-spot trace grid (hotspots highlighted)
- Response map image
- Opto vs. baseline trace overlay
- AUC and response-rate scatter-bar plots

In [ ]:
search_result = ndap.analyze_dmd_search(
    cells_df,
    cell=3,
    search_idx=0,
    make_plots=True,
    save_dir=SAVE_DIR,
)

print(f"Depths analysed: {len(search_result['depth_results'])}")
for dr in search_result["depth_results"]:
    n_hs = dr["hotspot_spot_idx"].sum()
    n_tot = len(dr["hotspot_spot_idx"])
    print(f"  depth {dr['depth']}: {n_hs}/{n_tot} hotspots")

In [ ]:
# Per-depth metrics summary
metrics = search_result["metrics_df"]
metrics.groupby("depth")[["auc", "ctrl_auc", "e_rate", "i_rate"]].mean().round(4)

In [ ]:
# Display the figure for depth 1
search_result["figures"][0]

## 5. Paired search comparison — `analyze_dmd_search_pair`

Cell 2 has two searches (`epoch8` and `epoch9`), which form one pair.
This computes per-common-depth AUC differences and comparison figures.

In [ ]:
pair_result = ndap.analyze_dmd_search_pair(
    cells_df,
    cell=2,
    pair_idx=0,
    make_plots=True,
    save_dir=SAVE_DIR,
)

print(f"Common depths compared: {len(pair_result['depth_results'])}")
for dr in pair_result["depth_results"]:
    print(f"  depth {dr['depth']}: opto1={dr['opto1'].shape}, opto2={dr['opto2'].shape}")

In [ ]:
# Per-depth AUC difference summary
pair_metrics = pair_result["metrics_df"]
pair_metrics.groupby("depth")[["auc_search1", "auc_search2", "auc_diff"]].mean().round(4)

In [ ]:
# Display depth-1 pair comparison figure
pair_result["figures"][0]

In [ ]:
plt.close("all")
print(f"Done — figures saved to:\n  {SAVE_DIR}")